# ETH model pipeline

In [1]:
# Load Packages
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
data = pd.read_csv('eth_ml_dataset.csv')
data.head(1)

,date,BTC,VIX,ETH,bitcoin,cryptocurrency,ethereum,crypto,btc,eth,crypto crash,bitcoin crash,buy bitcoin,sell bitcoin,crypto bull run,solana,altcoin,bitcoin price,crypto wallet
0,2018-01-02,14982.099609,9.77,884.44397,52,10,8,7,11,67,0,3,42,5,0,1,2,49,1


In [3]:
# --- Time-series target: predict *tomorrow* from *today's* info (no same-day ETH in features) ---
df = data.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Next-day simple return: (ETH tomorrow / ETH today) - 1
df["y_eth_next_return"] = df["ETH"].shift(-1) / df["ETH"] - 1

# Features: numeric columns only — no ETH, no date
feature_cols = [c for c in df.columns if c not in ("date", "ETH", "y_eth_next_return")]
X = df[feature_cols]
y = df["y_eth_next_return"]

# Last day has no "tomorrow" → drop NaN target
mask = y.notna()
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

# Chronological train / validation / test (no shuffle)
# Use validation for tuning (e.g. hyperparameters); use test only once for final metrics.
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.6, 0.2, 0.2
n = len(X)
n_test = int(n * TEST_FRAC)
n_val = int(n * VAL_FRAC)
n_train = n - n_val - n_test  # remainder avoids rounding gaps

X_train = X.iloc[:n_train]
y_train = y.iloc[:n_train]
X_val = X.iloc[n_train : n_train + n_val]
y_val = y.iloc[n_train : n_train + n_val]
X_test = X.iloc[n_train + n_val :]
y_test = y.iloc[n_train + n_val :]

# Scale using train statistics only, then apply to val and test
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Train:", len(y_train), "| Val:", len(y_val), "| Test:", len(y_test), "| Total:", n)
print("Feature count:", X_train.shape[1])

Train: 1245 | Val: 415 | Test: 415 | Total: 2075
Feature count: 17


In [4]:
# Define models and hyperparameter grid (fresh estimators per family for GridSearchCV)
from sklearn.base import clone
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LinearRegression

MODEL_BUILDERS = {
    "Lasso": lambda: Lasso(),
    "Ridge": lambda: Ridge(),
    "ElasticNet": lambda: ElasticNet(),
}

param_grid = {
    "Lasso": {
        "model__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000],
        "model__max_iter": [15000],
    },
    "Ridge": {"model__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000]},
    "ElasticNet": {
        "model__alpha": [0.01, 0.1, 1, 10],
        "model__l1_ratio": [0.2, 0.5, 0.8],
        "model__max_iter": [15000],
    },
}

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# Time-series CV: each fold trains only on the past (causal splits)
tscv = TimeSeriesSplit(n_splits=15)
best_model = {}

for name, builder in MODEL_BUILDERS.items():
    pipeline = Pipeline([("model", builder())])
    grid_search = GridSearchCV(
        pipeline,
        param_grid[name],
        cv=tscv,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
    )
    grid_search.fit(X_train, y_train)
    best_model[name] = grid_search.best_estimator_
    print(f"Best parameters for {name}: {grid_search.best_params_}")

# Choose model family by validation RMSE (test set stays untouched until the end)
val_rmse = {}
for name, est in best_model.items():
    pred_val = est.predict(X_val)
    val_rmse[name] = float(np.sqrt(mean_squared_error(y_val, pred_val)))
    print(f"{name} RMSE on validation (next-day return): {val_rmse[name]:.6f}")

selected_name = min(val_rmse, key=val_rmse.get)
print(f"\nSelected model (lowest validation RMSE): {selected_name}")

# Refit selected pipeline on train + validation before final test (more data, same hyperparameters)
final_estimator = clone(best_model[selected_name])
X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])
final_estimator.fit(X_trainval, y_trainval)

Best parameters for Lasso: {'model__alpha': 0.01, 'model__max_iter': 15000}
Best parameters for Ridge: {'model__alpha': 10000}
Best parameters for ElasticNet: {'model__alpha': 0.01, 'model__l1_ratio': 0.5, 'model__max_iter': 15000}
Lasso RMSE on validation (next-day return): 0.036438
Ridge RMSE on validation (next-day return): 0.036413
ElasticNet RMSE on validation (next-day return): 0.036438

Selected model (lowest validation RMSE): Ridge


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",10000
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable

In [6]:
from sklearn.metrics import mean_squared_error

# Final test metric: only the model selected on validation (evaluated once on held-out test)
y_test_pred = final_estimator.predict(X_test)
test_rmse = float(np.sqrt(mean_squared_error(y_test, y_test_pred)))
print(f"Final TEST RMSE — {selected_name} (next-day return): {test_rmse:.6f}")

# Baselines: fit on train only, compare on validation (does not use test)
baselines = {
    "LinearRegression": LinearRegression(),
    "Lasso(alpha=1)": Lasso(alpha=1),
    "Ridge(alpha=1)": Ridge(alpha=1),
}
print("\nBaseline validation RMSE (for reference, default hyperparameters):")
for label, est in baselines.items():
    est.fit(X_train, y_train)
    pred_v = est.predict(X_val)
    rmse_v = float(np.sqrt(mean_squared_error(y_val, pred_v)))
    print(f"  {label}: {rmse_v:.6f}")

Final TEST RMSE — Ridge (next-day return): 0.044544

Baseline validation RMSE (for reference, default hyperparameters):
  LinearRegression: 0.038267
  Lasso(alpha=1): 0.036438
  Ridge(alpha=1): 0.038223


In [7]:
# Constant baselines on the same TEST set (apples-to-apples with Final TEST RMSE above)
# - Predict 0: no free parameters (common benchmark for returns)
# - Predict mean(y): best constant in MSE sense; use mean from train+val only (same info as final refit)

y_trainval_arr = np.concatenate([y_train, y_val])
mu_trainval = float(np.mean(y_trainval_arr))

pred_zero = np.zeros_like(y_test, dtype=float)
pred_mean = np.full_like(y_test, fill_value=mu_trainval, dtype=float)

rmse_zero = float(np.sqrt(mean_squared_error(y_test, pred_zero)))
rmse_mean = float(np.sqrt(mean_squared_error(y_test, pred_mean)))

print("TEST RMSE — constant models (next-day return):")
print(f"  Always 0:                {rmse_zero:.6f}")
print(f"  Always mean(train+val): {rmse_mean:.6f}  (mean = {mu_trainval:.6f})")
print(f"  Your model ({selected_name}): {test_rmse:.6f}")
if test_rmse < rmse_zero and test_rmse < rmse_mean:
    print("  → Lower RMSE than both constant baselines on this test period.")
elif test_rmse < rmse_mean:
    print("  → Lower RMSE than mean baseline (not necessarily than 0).")
else:
    print("  → Does not beat the constant baselines on RMSE for this test period.")

TEST RMSE — constant models (next-day return):
  Always 0:                0.044538
  Always mean(train+val): 0.044566  (mean = 0.002235)
  Your model (Ridge): 0.044544
  → Lower RMSE than mean baseline (not necessarily than 0).


### Simple backtest (test period only)

Long ETH when the model predicts **positive** next-day return; otherwise **cash** (0%). Compare to **buy-and-hold** (always long).  
Does not include fees or slippage — add a cost per trade if you need realism.

In [8]:
# Backtest on held-out TEST only (same predictions as Final TEST RMSE)
y_pred_test = final_estimator.predict(X_test)
y_act = np.asarray(y_test, dtype=float)

# Long-only rule: in the market when predicted next-day return > 0
pos_model = (y_pred_test > 0).astype(float)
strat_ret = pos_model * y_act

# Buy-and-hold: always invested
bh_ret = y_act

def total_return(r):
    r = np.asarray(r, dtype=float)
    return float(np.prod(1.0 + r) - 1.0)

def max_drawdown(wealth):
    """wealth: cumulative product path starting at 1.0"""
    peak = np.maximum.accumulate(wealth)
    dd = (wealth - peak) / peak
    return float(dd.min())

w_strat = np.cumprod(1.0 + strat_ret)
w_bh = np.cumprod(1.0 + bh_ret)

print("TEST period — cumulative metrics (simple, no fees):")
print(f"  Model strategy (long if pred>0) total return: {100 * total_return(strat_ret):.2f}%")
print(f"  Buy-and-hold ETH total return:               {100 * total_return(bh_ret):.2f}%")
print(f"  Max drawdown (strategy): {100 * max_drawdown(w_strat):.2f}%")
print(f"  Max drawdown (buy-hold):   {100 * max_drawdown(w_bh):.2f}%")
print(f"  Time in market (strategy): {100 * pos_model.mean():.1f}%")

TEST period — cumulative metrics (simple, no fees):
  Model strategy (long if pred>0) total return: -16.46%
  Buy-and-hold ETH total return:               -16.46%
  Max drawdown (strategy): -63.24%
  Max drawdown (buy-hold):   -63.24%
  Time in market (strategy): 100.0%
